# 🔧 Notebook 2: Data Preprocessing

**Môn học:** Học Máy (CO3001) — HK I (2025-2026)  
**Nhóm:** Hoàng Xuân Bách · Nguyễn Việt Hùng · Trần Văn Hùng

---
### Mục tiêu
- Xử lý missing values (Mean / Median / KNN Imputer)
- Encoding categorical features (One-Hot / Label / Target Encoding)
- Feature scaling (Standard / MinMax / Robust)
- So sánh các cấu hình preprocessing

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import joblib, warnings

warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from modules import config
from modules.preprocessing import DataPreprocessor, remove_duplicates, check_missing_values

print('✅ Libraries loaded!')

## 2. Load Data

In [ ]:
# TODO: Thay bằng dataset thật của bạn
# df = pd.read_csv('../data/raw/your_dataset.csv')

# ── Demo dataset ──
from sklearn.datasets import make_classification
X_demo, y_demo = make_classification(n_samples=1000, n_features=15,
                                     n_informative=10, n_redundant=3,
                                     random_state=42)
df = pd.DataFrame(X_demo, columns=[f'feature_{i}' for i in range(15)])
df['target'] = y_demo
np.random.seed(42)
df.loc[np.random.choice(df.index, 80), 'feature_0'] = np.nan
df.loc[np.random.choice(df.index, 50), 'feature_1'] = np.nan
df['category_col'] = np.random.choice(['A','B','C'], size=len(df))

print(f'Loaded: {df.shape}')  
df.head()

## 3. Xử Lý Duplicates

In [ ]:
df_clean = remove_duplicates(df, verbose=True)
print(f'Shape sau khi xóa duplicate: {df_clean.shape}')

## 4. Tách Features và Target

In [ ]:
target_col = 'target'  # TODO: Sửa tên cột target

X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print(f'Features shape: {X.shape}')
print(f'Target shape  : {y.shape}')
print(f'\nTarget classes: {sorted(y.unique())}')

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=y
)

print(f'Train : {X_train.shape}')
print(f'Test  : {X_test.shape}')
print(f'\nClass ratio (train):')
print((y_train.value_counts(normalize=True)*100).round(2))

## 6. So Sánh Các Cấu Hình Preprocessing

### Config 1: Mean Imputation + One-Hot Encoding + Standard Scaling

In [ ]:
prep1 = DataPreprocessor(
    imputation_method='mean',
    encoding_method='onehot',
    scaling_method='standard',
    verbose=True
)

X_train_1 = prep1.fit_transform(X_train, y_train)
X_test_1  = prep1.transform(X_test)

print(f'\nShape sau xử lý: train={X_train_1.shape}, test={X_test_1.shape}')

### Config 2: Median Imputation + Label Encoding + MinMax Scaling

In [ ]:
prep2 = DataPreprocessor(
    imputation_method='median',
    encoding_method='label',
    scaling_method='minmax',
    verbose=True
)

X_train_2 = prep2.fit_transform(X_train, y_train)
X_test_2  = prep2.transform(X_test)

### Config 3: KNN Imputation + Target Encoding + Robust Scaling

In [ ]:
prep3 = DataPreprocessor(
    imputation_method='knn',
    encoding_method='target',
    scaling_method='robust',
    verbose=True
)

X_train_3 = prep3.fit_transform(X_train, y_train)
X_test_3  = prep3.transform(X_test)

## 7. Kiểm Tra Kết Quả Preprocessing

In [ ]:
configs = {
    'Mean+OneHot+Standard' : X_train_1,
    'Median+Label+MinMax'  : X_train_2,
    'KNN+Target+Robust'    : X_train_3,
}

print('=== Preprocessing Results ===')
for name, X in configs.items():
    print(f'{name}: shape={X.shape}, ')

# Visualize phân phối trước/sau scaling (feature_0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(X_train['feature_0'].dropna(), bins=30, color='salmon', edgecolor='white')
axes[0].set_title('Before Scaling: feature_0', fontweight='bold')

axes[1].hist(X_train_1.iloc[:, 0], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('After Standard Scaling: feature_0', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/before_after_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Lưu Preprocessed Data

*(Chọn config tốt nhất sau bước training ở Notebook 3)*

In [ ]:
# Lưu config 1 làm default (thay đổi nếu cần)
chosen_X_train = X_train_1
chosen_X_test  = X_test_1
chosen_prep    = prep1

np.save(config.FEATURES_DIR / 'X_train.npy', chosen_X_train.values)
np.save(config.FEATURES_DIR / 'X_test.npy',  chosen_X_test.values)
np.save(config.FEATURES_DIR / 'y_train.npy', y_train.values)
np.save(config.FEATURES_DIR / 'y_test.npy',  y_test.values)

with open(config.FEATURES_DIR / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(chosen_prep.feature_names))

joblib.dump(chosen_prep, config.MODELS_DIR / 'preprocessor.pkl')

print('✅ Đã lưu preprocessed data!')
print(f'   X_train: {chosen_X_train.shape}')
print(f'   X_test : {chosen_X_test.shape}')
print(f'   Features: {len(chosen_prep.feature_names)}')